<img src="../images/cads-logo.png" style="height: 100px;" align=left> 
<img src="../images/sklearn-logo.png" style="height: 100px;" align=right>

# Supervised Machine Learning

# Table of Contents

- [Thinking about Model Validation](#Thinking-about-Model-Validation)
- [Cross Validation](#Cross-Validation)
- [Model validation the wrong way](#Model-validation-the-wrong-way)
    - [Question: Can you guess the result of the following cell?](#Question:-Can-you-guess-the-result-of-the-following-cell?)
- [Model validation the right way: Holdout sets](#Model-validation-the-right-way:-Holdout-sets)
- [Model validation via cross-validation](#Model-validation-via-cross-validation)
- [Grid Search](#Grid-Search)

# Thinking about Model Validation

In principle, model validation is very simple: after choosing a model and its hyperparameters, we can estimate how effective it is by applying it to some of the training data and comparing the prediction to the known value.

The following sections first show a naive approach to model validation and why it
fails, before exploring the use of holdout sets and cross-validation for more robust
model evaluation.

# Cross Validation

## Model validation the wrong way

Let's demonstrate the naive approach to validation using the Iris data, which we saw in the previous section. We will start by loading the data:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns; sns.set()

In [ ]:
from sklearn.datasets import load_iris
iris = load_iris()
X = iris.data
y = iris.target

print('Shape of X:', X.shape)
print('Shape of y:', y.shape)

Next we choose a model and hyperparameters. Here we'll use a *k*-neighbors classifier with ``n_neighbors=1``.
This is a very simple and intuitive model that says "the label of an unknown point is the same as the label of its closest training point:"

<img src='../images/KNN.png'>

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
model= KNeighborsClassifier(n_neighbors=1)

Then we train the model, and use it to predict labels for data we already know:

In [ ]:
model.fit(X, y)
y_model = model.predict(X)

Finally, we compute the fraction of correctly labeled points:

### Question: Can you guess the result of the following cell?

In [ ]:
from sklearn.metrics import accuracy_score
accuracy_score(y, y_model)

We see an accuracy score of 1.0, which indicates that 100% of points were correctly labeled by our model!
But is this truly measuring the expected accuracy? Have we really come upon a model that we expect to be correct 100% of the time?

As you may have gathered, the answer is no.
In fact, this approach contains a fundamental flaw: *it trains and evaluates the model on the same data*.
Furthermore, the nearest neighbor model is an *instance-based* estimator that simply stores the training data, and predicts labels by comparing new data to these stored points: except in contrived cases, it will get 100% accuracy *every time!*

## Model validation the right way: Holdout sets

So what can be done?
A better sense of a model's performance can be found using what's known as a *holdout set*: that is, we hold back some subset of the data from the training of the model, and then use this holdout set to check the model performance.
This splitting can be done using the ``train_test_split`` utility in Scikit-Learn:

In [ ]:
print(X.shape)
print(y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

# split the data with 50% in each set
X_train, X_test, y_train, y_test = train_test_split(X, y,random_state=0,train_size=0.5)

#fit the model
model.fit(X_train, y_train)

# fit and evaluate the model on the second set of data
y_model = model.predict(X_test)

accuracy_score(y_test, y_model)

In [ ]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

We see here a more reasonable result: the nearest-neighbor classifier is about 90% accurate on this hold-out set.
The hold-out set is similar to unknown data, because the model has not "seen" it before.

## Model validation via cross-validation

One disadvantage of using a holdout set for model validation is that we have lost a portion of our data to the model training.
In the above case, half the dataset does not contribute to the training of the model!
This is not optimal, and can cause problems – especially if the initial set of training data is small.

One way to address this is to use *cross-validation*; that is, to do a sequence of fits where each subset of the data is used both as a training set and as a validation set. Visually, it might look something like this:
<img src= "../images/2-fold-CV.png" style="height: 500px;">


Here we do two validation trials, alternately using each half of the data as a holdout set.
Using the split data from before, we could implement it like this:

**Question:**
Write the code that implements the accuracy described on the previous image

In [ ]:
# solution
y1_model = model.fit(X_train, y_train).predict(X_test)
y2_model = model.fit(X_test, y_test).predict(X_train)
accuracy_score(y_test, y1_model), accuracy_score(y_train, y2_model)

What comes out are two accuracy scores, which we could combine (by, say, taking the mean) to get a better measure of the global model performance.
This particular form of cross-validation is a *two-fold cross-validation*—that is, one in which we have split the data into two sets and used each in turn as a validation set.

We could expand on this idea to use even more trials, and more folds in the data—for example, here is a visual depiction of five-fold cross-validation:

<img src='../images/CV.png'/>

Here we split the data into five groups, and use each of them in turn to evaluate the model fit on the other 4/5 of the data.
This would be rather tedious to do by hand, and so we can use Scikit-Learn's ``cross_val_score`` convenience routine to do it succinctly:

In [ ]:
from sklearn.model_selection import cross_val_score

model= KNeighborsClassifier(n_neighbors=3)
scores = cross_val_score(model, X, y, cv=2)  
scores

In [ ]:
print("Accuracy: {}".format(scores.mean()))

By default, the score computed at each cv iteration is the `score` method of the estimator. It is possible to change this by using the scoring parameter. Take a look at all possible values for [scoring parameter](https://scikit-learn.org/stable/modules/model_evaluation.html).


In [ ]:
scores_f1 = cross_val_score(model, X, y, cv=2, scoring='f1_macro')   
scores_f1, np.mean(scores_f1)

Repeating the validation across different subsets of the data gives us an even better idea of the performance of the algorithm.

Scikit-Learn implements a number of useful cross-validation schemes that are useful in particular situations; these are implemented via iterators in the ``cross_validation`` module.
For example, we might wish to go to the extreme case in which our number of folds is equal to the number of data points: that is, we train on all points but one in each trial.
This type of cross-validation is known as *leave-one-out* cross validation, and can be used as follows:

In [ ]:
from sklearn.model_selection import LeaveOneOut

model= KNeighborsClassifier(n_neighbors=3)
scores = cross_val_score(model, X, y, cv=LeaveOneOut())
scores

Because we have 150 samples, the leave one out cross-validation yields scores for 150 trials, and the score indicates either successful (1.0) or unsuccessful (0.0) prediction.
Taking the mean of these gives an estimate of the error rate:

In [ ]:
scores.mean()

Other cross-validation schemes can be used similarly.
For a description of what is available in Scikit-Learn, use IPython to explore the ``sklearn.cross_validation`` submodule, or take a look at Scikit-Learn's online [cross-validation documentation](http://scikit-learn.org/stable/modules/cross_validation.html).

**Exercise:**
Try to classify Iris data using KNN for n_neighbors=4. Use 5-fold cross validation and use Accuracy, Precision, Recall, F1-score as evaluation metrics.

In [4]:
from sklearn.datasets import load_iris
from sklearn.model_selection import cross_validate
from sklearn.neighbors import KNeighborsClassifier

iris = load_iris()
X = iris.data
y = iris.target

knn4 = KNeighborsClassifier(n_neighbors=4)
scoring = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']

cv_results = cross_validate(knn4, X, y, cv=5, scoring=scoring)

print('Accuracy :', cv_results['test_accuracy'].mean())
print('Precision:', cv_results['test_precision_macro'].mean())
print('Recall   :', cv_results['test_recall_macro'].mean())
print('F1-score :', cv_results['test_f1_macro'].mean())

Accuracy : 0.9733333333333334
Precision: 0.9757575757575758
Recall   : 0.9733333333333334
F1-score : 0.9732664995822891


### Exercise
Try to classify 'indian_liver_patient.csv' data using KNN for n_neighbors=5 and Logistic Regression. Use 3-fold cross validation.

This data set contains 416 liver patient records and 167 non liver patient records collected from North East of Andhra Pradesh, India. The "Dataset" column is a class label used to divide groups into liver patient (liver disease) or not (no disease). 

In [5]:
import pandas as pd

liver = pd.read_csv("../Data/indian_liver_patient.csv")
liver.head(2)

,Age,Gender,Total_Bilirubin,Direct_Bilirubin,Alkaline_Phosphotase,Alamine_Aminotransferase,Aspartate_Aminotransferase,Total_Protiens,Albumin,Albumin_and_Globulin_Ratio,Dataset
0,65,Female,0.7,0.1,187,16,18,6.8,3.3,0.90,0
1,62,Male,10.9,5.5,699,64,100,7.5,3.2,0.74,0


In [6]:
liver.info()

<class 'pandas.DataFrame'>
RangeIndex: 583 entries, 0 to 582
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Age                         583 non-null    int64  
 1   Gender                      583 non-null    str    
 2   Total_Bilirubin             583 non-null    float64
 3   Direct_Bilirubin            583 non-null    float64
 4   Alkaline_Phosphotase        583 non-null    int64  
 5   Alamine_Aminotransferase    583 non-null    int64  
 6   Aspartate_Aminotransferase  583 non-null    int64  
 7   Total_Protiens              583 non-null    float64
 8   Albumin                     583 non-null    float64
 9   Albumin_and_Globulin_Ratio  583 non-null    float64
 10  Dataset                     583 non-null    int64  
dtypes: float64(5), int64(5), str(1)
memory usage: 50.2 KB


In [8]:
#Assuming 0=no disease and 1=disease
liver.Dataset.value_counts()

Dataset
0    416
1    167
Name: count, dtype: int64

In [9]:
X = liver.drop('Dataset', axis=1)
y = liver.Dataset

In [10]:
X_dum=pd.get_dummies(X, drop_first=True)
X_dum

,Age,Total_Bilirubin,Direct_Bilirubin,Alkaline_Phosphotase,Alamine_Aminotransferase,Aspartate_Aminotransferase,Total_Protiens,Albumin,Albumin_and_Globulin_Ratio,Gender_Male
0,65,0.7,0.1,187,16,18,6.8,3.3,0.90,False
1,62,10.9,5.5,699,64,100,7.5,3.2,0.74,True
2,62,7.3,4.1,490,60,68,7.0,3.3,0.89,True
3,58,1.0,0.4,182,14,20,6.8,3.4,1.00,True
4,72,3.9,2.0,195,27,59,7.3,2.4,0.40,True
...,...,...,...,...,...,...,...,...,...,...
578,60,0.5,0.1,500,20,34,5.9,1.6,0.37,True
579,40,0.6,0.1,98,35,31,6.0,3.2,1.10,True
580,52,0.8,0.2,245,48,49,6.4,3.2,1.00,True
581,31,1.3,0.5,184,29,32,6.8,3.4,1.00,True


In [12]:
### KNN

import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler

x_scaled = MinMaxScaler().fit_transform(X_dum)

knn_5 = KNeighborsClassifier(n_neighbors=5)

A_scores = cross_val_score(knn_5, x_scaled, y, cv=3, scoring='precision_macro')
A_mean = np.mean(A_scores)
print('Mean of precision: {}'.format(A_mean))

Mean of precision: 0.5797898665311592


In [15]:
### Logistic Regression
from sklearn.linear_model import LogisticRegression

lr_5= LogisticRegression()

A_scores=cross_val_score(lr_5, x_scaled, y, cv=3, scoring='precision_macro')
A_mean=np.mean(A_scores)
print('Mean of precision: {}'.format(A_mean))

ValueError: Found input variables with inconsistent numbers of samples: [583, 569]

# Grid Search
Grid search is the process of performing hyper parameter tuning in order to determine the optimal values for a given model. Scikit-Learn provides automated tools to do this in the grid search module.

Here is an example of using grid search to find the optimal KNN model. This can be set up using Scikit-Learn's ``GridSearchCV`` meta-estimator:

Now let's load breast_cancer data that is available from sklearn datasets.

In [13]:
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer()

X = cancer.data
y= cancer.target
X.shape

(569, 30)

In [16]:
np.sum(y)

np.int64(357)

In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, train_size=0.80, stratify=y)

In [18]:
from sklearn.model_selection import GridSearchCV

param_grid = {'n_neighbors': np.arange(1, 20),
              'p': [1,2],
              'weights': ['uniform','distance']}

grid = GridSearchCV(KNeighborsClassifier(), 
                    param_grid, scoring='precision_macro',
                    cv=3)

In [ ]:
# What is the total number of models that grid search builds


Notice that like a normal estimator, this has not yet been applied to any data.
Calling the ``fit()`` method will fit the model at each grid point, keeping track of the scores along the way:

In [ ]:
grid.fit(X_train, y_train)

Now that this is fit, we can ask for the best parameters as follows:

In [ ]:
grid.best_params_

In [ ]:
grid

Finally, if we wish, we can use the best model and show the fit to our data using code from before:

In [ ]:
model = grid.best_estimator_
model.fit(X_train,y_train)
y_pred=model.predict(X_test)

from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report
print('classification_report:\n',classification_report(y_test, y_pred))
confusion_matrix(y_test, y_pred)

The grid search provides many more options, including the ability to specify a custom scoring function, to parallelize the computations, to do randomized searches, and more.

## Exercise:

Load the cancer dataset and choose the best classification algorithm with the best hyperparameters.

- Define X and y

- To simplify, remove missing values

- Split data to train and test

- Use 5 fold cross validation and grid search on train data

- Choose appropriate validation metric

- Set grid parameters for each classification algorithm

- Build the best models for each classification algorithm according to the best estimator (best hyperparameters) given by the grid search

- Compare the performance of the algorithms with the best hyperparametrs on the test data according to confusion matrix, recall, precision, F1, and auc metrics

In [27]:
df=pd.read_csv('../data/breast_cancer_wisconsin.csv')
df

,Id,Cl.thickness,Cell.size,Cell.shape,Marg.adhesion,Epith.c.size,Bare.nuclei,Bl.cromatin,Normal.nucleoli,Mitoses,Class
0,1000025,5,1,1,1,2,1.0,3,1,1,0
1,1002945,5,4,4,5,7,10.0,3,2,1,0
2,1015425,3,1,1,1,2,2.0,3,1,1,0
3,1016277,6,8,8,1,3,4.0,3,7,1,0
4,1017023,4,1,1,3,2,1.0,3,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...
694,776715,3,1,1,1,3,2.0,1,1,1,0
695,841769,2,1,1,1,2,1.0,1,1,1,0
696,888820,5,10,10,3,7,3.0,8,10,2,1
697,897471,4,8,6,4,3,4.0,10,6,1,1


In [5]:
import numpy as np
import pandas as pd

# Load and clean data
df = pd.read_csv('../Data/breast_cancer_wisconsin.csv', na_values=['NA', '?'])
df = df.replace('?', np.nan).dropna().copy()

# Define X and y
X = df.drop(columns=['Class', 'Id']).apply(pd.to_numeric, errors='coerce')
y = pd.to_numeric(df['Class'], errors='coerce')

valid_rows = X.notna().all(axis=1) & y.notna()
X = X.loc[valid_rows]
y = y.loc[valid_rows].astype(int)

print('X shape:', X.shape)
print('y shape:', y.shape)
print('Class counts:\n', y.value_counts())

X shape: (683, 9)
y shape: (683,)
Class counts:
 Class
0    444
1    239
Name: count, dtype: int64


In [6]:
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, recall_score, precision_score, f1_score, roc_auc_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# Split data to train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Use F1 because the classes are not perfectly balanced
scoring_metric = 'f1'

# Helper for test-set evaluation
def evaluate_on_test(model_name, model):
    y_pred = model.predict(X_test)
    if hasattr(model, 'predict_proba'):
        y_score = model.predict_proba(X_test)[:, 1]
    else:
        y_score = model.decision_function(X_test)

    return {
        'Model': model_name,
        'Confusion Matrix': confusion_matrix(y_test, y_pred).tolist(),
        'Recall': recall_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'AUC': roc_auc_score(y_test, y_score)
    }

print('X_train:', X_train.shape)
print('X_test :', X_test.shape)

X_train: (546, 9)
X_test : (137, 9)


In [7]:
knn_grid = GridSearchCV(
    Pipeline([
        ('scaler', StandardScaler()),
        ('model', KNeighborsClassifier())
    ]),
    {
        'model__n_neighbors': [3, 5, 7, 9, 11],
        'model__weights': ['uniform', 'distance'],
        'model__p': [1, 2]
    },
    cv=5,
    scoring=scoring_metric,
    n_jobs=-1
)
knn_grid.fit(X_train, y_train)
knn_grid.best_score_, knn_grid.best_params_

(np.float64(0.9685407834268593),
 {'model__n_neighbors': 7, 'model__p': 2, 'model__weights': 'uniform'})

In [8]:
dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    {
        'max_depth': [None, 3, 5, 7, 10],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    },
    cv=5,
    scoring=scoring_metric,
    n_jobs=-1
)
dt_grid.fit(X_train, y_train)
dt_grid.best_score_, dt_grid.best_params_

(np.float64(0.9395187292555714),
 {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 5})

In [9]:
lr_grid = GridSearchCV(
    Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=3000))
    ]),
    {
        'model__C': [0.01, 0.1, 1, 10, 100],
        'model__solver': ['liblinear', 'lbfgs']
    },
    cv=5,
    scoring=scoring_metric,
    n_jobs=-1
)
lr_grid.fit(X_train, y_train)
lr_grid.best_score_, lr_grid.best_params_

(np.float64(0.9583564452425211),
 {'model__C': 0.01, 'model__solver': 'liblinear'})

In [10]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

scoring_metric = globals().get('scoring_metric', 'f1')

if 'X_train' not in globals() or 'y_train' not in globals():
    if 'X' not in globals() or 'y' not in globals():
        df = pd.read_csv('../Data/breast_cancer_wisconsin.csv', na_values=['NA', '?'])
        df = df.replace('?', np.nan).dropna().copy()
        X = df.drop(columns=['Class', 'Id']).apply(pd.to_numeric, errors='coerce')
        y = pd.to_numeric(df['Class'], errors='coerce')
        valid_rows = X.notna().all(axis=1) & y.notna()
        X = X.loc[valid_rows]
        y = y.loc[valid_rows].astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

svm_grid = GridSearchCV(
    Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(probability=True, random_state=42))
    ]),
    {
        'model__C': [0.1, 1, 10, 100],
        'model__kernel': ['linear', 'rbf'],
        'model__gamma': ['scale', 'auto']
    },
    cv=5,
    scoring=scoring_metric,
    n_jobs=-1
)
svm_grid.fit(X_train, y_train)
svm_grid.best_score_, svm_grid.best_params_

c:\Users\TM39104\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


(np.float64(0.9556906160503763),
 {'model__C': 0.1, 'model__gamma': 'scale', 'model__kernel': 'linear'})

In [11]:
cv_summary_df = pd.DataFrame([
    {'Model': 'KNN', 'CV Best F1': knn_grid.best_score_, 'Best Params': knn_grid.best_params_},
    {'Model': 'Decision Tree', 'CV Best F1': dt_grid.best_score_, 'Best Params': dt_grid.best_params_},
    {'Model': 'Logistic Regression', 'CV Best F1': lr_grid.best_score_, 'Best Params': lr_grid.best_params_},
    {'Model': 'SVM', 'CV Best F1': svm_grid.best_score_, 'Best Params': svm_grid.best_params_}
]).sort_values(by='CV Best F1', ascending=False)

cv_summary_df

,Model,CV Best F1,Best Params
0,KNN,0.968541,"{'model__n_neighbors': 7, 'model__p': 2, 'mode..."
2,Logistic Regression,0.958356,"{'model__C': 0.01, 'model__solver': 'liblinear'}"
3,SVM,0.955691,"{'model__C': 0.1, 'model__gamma': 'scale', 'mo..."
1,Decision Tree,0.939519,"{'max_depth': None, 'min_samples_leaf': 1, 'mi..."


In [12]:
best_models = {
    'KNN': knn_grid.best_estimator_,
    'Decision Tree': dt_grid.best_estimator_,
    'Logistic Regression': lr_grid.best_estimator_,
    'SVM': svm_grid.best_estimator_
}

best_models

{'KNN': Pipeline(steps=[('scaler', StandardScaler()),
                 ('model', KNeighborsClassifier(n_neighbors=7))]),
 'Decision Tree': DecisionTreeClassifier(min_samples_split=5, random_state=42),
 'Logistic Regression': Pipeline(steps=[('scaler', StandardScaler()),
                 ('model',
                  LogisticRegression(C=0.01, max_iter=3000,
                                     solver='liblinear'))]),
 'SVM': Pipeline(steps=[('scaler', StandardScaler()),
                 ('model',
                  SVC(C=0.1, kernel='linear', probability=True,
                      random_state=42))])}

**Knn**

In [13]:
knn_metrics = evaluate_on_test('KNN', best_models['KNN'])

pd.Series({
    'Best Params': knn_grid.best_params_,
    'CV Best F1': knn_grid.best_score_,
    **knn_metrics
})

Best Params         {'model__n_neighbors': 7, 'model__p': 2, 'mode...
CV Best F1                                                   0.968541
Model                                                             KNN
Confusion Matrix                                   [[85, 4], [2, 46]]
Recall                                                       0.958333
Precision                                                        0.92
F1                                                           0.938776
AUC                                                          0.982795
dtype: object

**Decision Tree**

In [14]:
dt_metrics = evaluate_on_test('Decision Tree', best_models['Decision Tree'])

pd.Series({
    'Best Params': dt_grid.best_params_,
    'CV Best F1': dt_grid.best_score_,
    **dt_metrics
})

Best Params         {'max_depth': None, 'min_samples_leaf': 1, 'mi...
CV Best F1                                                   0.939519
Model                                                   Decision Tree
Confusion Matrix                                   [[85, 4], [2, 46]]
Recall                                                       0.958333
Precision                                                        0.92
F1                                                           0.938776
AUC                                                          0.955758
dtype: object

**Logistic Regression**

In [15]:
lr_metrics = evaluate_on_test('Logistic Regression', best_models['Logistic Regression'])

pd.Series({
    'Best Params': lr_grid.best_params_,
    'CV Best F1': lr_grid.best_score_,
    **lr_metrics
})

Best Params         {'model__C': 0.01, 'model__solver': 'liblinear'}
CV Best F1                                                  0.958356
Model                                            Logistic Regression
Confusion Matrix                                  [[85, 4], [1, 47]]
Recall                                                      0.979167
Precision                                                   0.921569
F1                                                          0.949495
AUC                                                          0.99368
dtype: object

**Support Vector Machine (SVM)**

In [16]:
svm_metrics = evaluate_on_test('SVM', best_models['SVM'])

pd.Series({
    'Best Params': svm_grid.best_params_,
    'CV Best F1': svm_grid.best_score_,
    **svm_metrics
})

Best Params         {'model__C': 0.1, 'model__gamma': 'scale', 'mo...
CV Best F1                                                   0.955691
Model                                                             SVM
Confusion Matrix                                   [[85, 4], [2, 46]]
Recall                                                       0.958333
Precision                                                        0.92
F1                                                           0.938776
AUC                                                          0.993212
dtype: object

**PipeLine: Polynomial Logistic Regression**

In [17]:
from sklearn.preprocessing import PolynomialFeatures

poly_lr_grid = GridSearchCV(
    Pipeline([
        ('scaler', StandardScaler()),
        ('poly', PolynomialFeatures(include_bias=False)),
        ('model', LogisticRegression(max_iter=5000))
    ]),
    {
        'poly__degree': [2, 3],
        'model__C': [0.01, 0.1, 1, 10],
        'model__solver': ['liblinear', 'lbfgs']
    },
    cv=5,
    scoring=scoring_metric,
    n_jobs=-1
)
poly_lr_grid.fit(X_train, y_train)

poly_lr_metrics = evaluate_on_test('Polynomial Logistic Regression', poly_lr_grid.best_estimator_)

pd.Series({
    'Best Params': poly_lr_grid.best_params_,
    'CV Best F1': poly_lr_grid.best_score_,
    **poly_lr_metrics
})

Best Params         {'model__C': 0.01, 'model__solver': 'liblinear...
CV Best F1                                                   0.963283
Model                                  Polynomial Logistic Regression
Confusion Matrix                                   [[85, 4], [1, 47]]
Recall                                                       0.979167
Precision                                                    0.921569
F1                                                           0.949495
AUC                                                          0.993914
dtype: object

In [18]:
all_results_df = pd.DataFrame([
    knn_metrics,
    dt_metrics,
    lr_metrics,
    svm_metrics,
    poly_lr_metrics
]).sort_values(by='F1', ascending=False)

all_results_df[['Model', 'Recall', 'Precision', 'F1', 'AUC']]

,Model,Recall,Precision,F1,AUC
2,Logistic Regression,0.979167,0.921569,0.949495,0.993680
4,Polynomial Logistic Regression,0.979167,0.921569,0.949495,0.993914
0,KNN,0.958333,0.920000,0.938776,0.982795
1,Decision Tree,0.958333,0.920000,0.938776,0.955758
3,SVM,0.958333,0.920000,0.938776,0.993212


**Compare the best models on the test data**

In [51]:
all_results_df[['Model', 'Confusion Matrix']]

,Model,Confusion Matrix
2,Logistic Regression,"[[85, 4], [1, 47]]"
4,Polynomial Logistic Regression,"[[85, 4], [1, 47]]"
0,KNN,"[[85, 4], [2, 46]]"
1,Decision Tree,"[[85, 4], [2, 46]]"
3,SVM,"[[85, 4], [2, 46]]"


In [52]:
all_results_df[['Model', 'Recall']].sort_values(by='Recall', ascending=False)

,Model,Recall
2,Logistic Regression,0.979167
4,Polynomial Logistic Regression,0.979167
0,KNN,0.958333
1,Decision Tree,0.958333
3,SVM,0.958333


In [53]:
all_results_df[['Model', 'Precision']].sort_values(by='Precision', ascending=False)

,Model,Precision
2,Logistic Regression,0.921569
4,Polynomial Logistic Regression,0.921569
0,KNN,0.920000
1,Decision Tree,0.920000
3,SVM,0.920000


In [54]:
all_results_df[['Model', 'F1']].sort_values(by='F1', ascending=False)

,Model,F1
2,Logistic Regression,0.949495
4,Polynomial Logistic Regression,0.949495
0,KNN,0.938776
1,Decision Tree,0.938776
3,SVM,0.938776


In [55]:
all_results_df[['Model', 'AUC']].sort_values(by='AUC', ascending=False)

,Model,AUC
4,Polynomial Logistic Regression,0.993914
2,Logistic Regression,0.993680
3,SVM,0.993212
0,KNN,0.982795
1,Decision Tree,0.955758


In [56]:
best_overall_model = all_results_df.sort_values(by='F1', ascending=False).iloc[0]
print('Best overall model:', best_overall_model['Model'])
print('Recall   :', best_overall_model['Recall'])
print('Precision:', best_overall_model['Precision'])
print('F1       :', best_overall_model['F1'])
print('AUC      :', best_overall_model['AUC'])
print('Confusion Matrix:', best_overall_model['Confusion Matrix'])

Best overall model: Logistic Regression
Recall   : 0.9791666666666666
Precision: 0.9215686274509803
F1       : 0.9494949494949495
AUC      : 0.9936797752808989
Confusion Matrix: [[85, 4], [1, 47]]
